# Capitolo 18 – Ricorsivo e iterativo a confronto (C++)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap18_ricorsivo_iterativo_cpp.ipynb)

Gli algoritmi del capitolo, ricorsivi e iterativi, a confronto con **conteggi** (chiamate, iterazioni, confronti, profondità della ricorsione) e con qualche tempo misurato: **SumExplicitStack** (la ricorsione simulata con una pila), **MergeSort** ricorsivo e **MergeSortBottomUp**, la potenza veloce ricorsiva e **PowerIt**, **Kadane**, i tre algoritmi di Fibonacci **FibRec**, **FibMemo** e **FibIt**, **GCDRec**/**GCDIt** e la ricerca binaria ricorsiva e iterativa (eliminazione della ricorsione in coda). I conteggi sono uguali in ogni linguaggio e in ogni macchina; i tempi (in ms) dipendono dalla macchina e dal linguaggio, contano solo come crescono.

Il capitolo completo, con pseudocodice, dimostrazioni ed esempi, è [sul sito](https://fabiofurini.github.io/algoritmi/analisi-ricorsione/18-ricorsivo-iterativo/).

Ogni programma è scritto in un file con la cella `%%writefile` e poi compilato ed eseguito con `g++ -std=c++17 -O2`: si eseguono le celle in ordine, dall'alto in basso. Nessuna libreria esterna.

**Indici.** Nello pseudocodice delle dispense gli array sono indicizzati da 1; nel codice (come in tutti i linguaggi qui usati) da 0: l'elemento `a[i]` dello pseudocodice è l'elemento di indice `i-1` nel codice, e un intervallo `a[l..r]` diventa `a[l-1..r-1]`. Le stampe dei notebook usano gli indici da 0.

**Nomi.** Le funzioni hanno il nome dello pseudocodice, nella convenzione del linguaggio (come nei capitoli di introduzione): `FibRec` → `FibRec` (PascalCase).

## 1. Dalla ricorsione a una pila esplicita: SumExplicitStack

**SumExplicitStack**$(a,n)$ — *Input*: un array $a[1..n]$ di $n\ge1$ interi. *Output*: la somma $a[1]+\dots+a[n]$. La versione iterativa della somma per dimezzamento: la pila $P$ contiene le coppie $(l,r)$ dei sotto-array ancora da sommare e fa il lavoro della pila delle chiamate. La pila non contiene mai più di $\Theta(\log_2 n)$ coppie.

Sull'array $a=[4,7,1,9]$ del capitolo si stampa la traccia (con gli indici da 0: la coppia $(1,4)$ del capitolo è qui $(0,3)$; in ogni riga la cima della pila è a destra) e si confronta con `SumIterative`, `sumRecHalving` e `sumRecDecrease` del codice del corso, con la profondità massima della ricorsione. Poi si stampa la pila delle chiamate di `sumRecDecrease(a, 3)` su $a=[4,7,1]$ (esempio del capitolo): le addizioni avvengono solo *dopo* il caso base, mentre la pila si svuota, e a un certo punto le chiamate attive sono $n+1$.

In [1]:
%%writefile explicit_stack.cpp
#include <iostream>
#include <vector>
#include <string>
#include <algorithm>
#include <cmath>
#include <cstdio>
#include <chrono>
#include <climits>
#include <functional>
#include <array>
using namespace std;

int maxP = 0;   // massima dimensione della pila P

typedef pair<int, int> Pair;

// SumExplicitStack(a, n): somma di a[0..n-1] con una pila esplicita
long long SumExplicitStack(int *a, int n)
{
    long long s = 0;
    vector<Pair> P;                 // pila dei sotto-array da sommare
    P.push_back({0, n - 1});
    maxP = max(maxP, (int)P.size());
    while (!P.empty())
    {
        Pair top = P.back();
        P.pop_back();
        int l = top.first, r = top.second;
        if (l == r)
        {
            s = s + a[l];           // caso base
        }
        else
        {
            int mid = l + (r - l) / 2;
            P.push_back({mid + 1, r});
            P.push_back({l, mid});
            maxP = max(maxP, (int)P.size());
        }
    }
    return s;
}

string FmtP(const vector<Pair> &P)
{
    string s = "[";
    for (size_t i = 0; i < P.size(); i++)
    {
        if (i > 0) s += ",";
        s += "(" + to_string(P[i].first) + "," + to_string(P[i].second) + ")";
    }
    return s + "]";
}

long long SumExplicitStackTrace(int *a, int n)
{
    long long s = 0;
    vector<Pair> P = {{0, n - 1}};
    while (!P.empty())
    {
        Pair top = P.back();
        P.pop_back();
        int l = top.first, r = top.second;
        if (l == r)
        {
            s = s + a[l];
            printf("pop (%d,%d): caso base, s=%lld => P=%s\n", l, r, s, FmtP(P).c_str());
        }
        else
        {
            int mid = l + (r - l) / 2;
            P.push_back({mid + 1, r});
            P.push_back({l, mid});
            printf("pop (%d,%d): mid=%d, push (%d,%d) e (%d,%d) => P=%s, s=%lld\n", l, r, mid, mid + 1, r, l, mid, FmtP(P).c_str(), s);
        }
    }
    return s;
}

// ---- le altre somme del codice del corso, con la profondità ----
int depth = 0;
int maxDepth = 0;

long long sumIt(int *a, int n)
{
    long long s = 0;
    for (int i = 0; i < n; i++)
    {
        s = s + a[i];
    }
    return s;
}

long long sumRecHalving(int *a, int l, int r)
{
    depth++;
    if (depth > maxDepth) maxDepth = depth;
    long long result;
    if (l == r)
    {
        result = a[l];
    }
    else
    {
        int mid = l + (r - l) / 2;
        result = sumRecHalving(a, l, mid) + sumRecHalving(a, mid + 1, r);
    }
    depth--;
    return result;
}

long long sumRecDecrease(int *a, int n)
{
    depth++;
    if (depth > maxDepth) maxDepth = depth;
    long long result;
    if (n == 0)
    {
        result = 0;
    }
    else
    {
        result = sumRecDecrease(a, n - 1) + a[n - 1];
    }
    depth--;
    return result;
}

// ---- la pila delle chiamate di sumRecDecrease(a, 3) ----
int stepNo = 0;
int maxActive = 0;

string Fmt(const int *a, int n)
{
    string s = "[";
    for (int i = 0; i < n; i++)
    {
        if (i > 0) s += ", ";
        s += to_string(a[i]);
    }
    return s + "]";
}

void Emit(const vector<string> &stack, const string &action)
{
    stepNo++;
    if ((int)stack.size() > maxActive) maxActive = stack.size();
    string st;
    for (size_t i = 0; i < stack.size(); i++)
    {
        if (i > 0) st += " ";
        st += stack[i];
    }
    printf("%2d | %s | %s\n", stepNo, st.c_str(), action.c_str());
}

long long SumRecDecreaseTrace(int *a, int n, vector<string> &stack)
{
    stack.push_back("s(" + to_string(n) + ")");
    if (n == 0)
    {
        Emit(stack, "n=0: return 0");
        stack.pop_back();
        return 0;
    }
    Emit(stack, "n=" + to_string(n) + " != 0: chiama s(" + to_string(n - 1) + "), poi somma a[" + to_string(n - 1) + "]");
    long long r = SumRecDecreaseTrace(a, n - 1, stack);
    Emit(stack, "return " + to_string(r) + "+a[" + to_string(n - 1) + "]=" + to_string(r + a[n - 1]));
    stack.pop_back();
    return r + a[n - 1];
}

#ifndef NOMAIN
int main()
{
    int a[] = {4, 7, 1, 9};
    int n = 4;
    printf("a = %s   n = %d\n", Fmt(a, n).c_str(), n);
    printf("SumExplicitStack(a, 4):\n");
    long long res = SumExplicitStackTrace(a, n);
    maxP = 0;
    SumExplicitStack(a, n);
    printf("risultato %lld, dimensione massima di P = %d\n", res, maxP);
    printf("\n");
    printf("sumIt                = %lld\n", sumIt(a, n));
    maxDepth = 0;
    long long r1 = sumRecHalving(a, 0, n - 1);
    printf("sumRecHalving        = %lld   profondità massima = %d\n", r1, maxDepth);
    maxDepth = 0;
    long long r2 = sumRecDecrease(a, n);
    printf("sumRecDecrease       = %lld   profondità massima = %d\n", r2, maxDepth);

    int b[] = {4, 7, 1};
    printf("\n");
    printf("a = %s   sumRecDecrease(a, 3)\n", Fmt(b, 3).c_str());
    printf(" # | pila delle chiamate attive | azione\n");
    vector<string> stack;
    res = SumRecDecreaseTrace(b, 3, stack);
    printf("risultato %lld, chiamate attive al massimo = %d = n+1\n", res, maxActive);
    return 0;
}
#endif

Writing explicit_stack.cpp


In [2]:
!g++ -std=c++17 -O2 -o explicit_stack explicit_stack.cpp && ./explicit_stack

a = [4, 7, 1, 9]   n = 4
SumExplicitStack(a, 4):
pop (0,3): mid=1, push (2,3) e (0,1) => P=[(2,3),(0,1)], s=0
pop (0,1): mid=0, push (1,1) e (0,0) => P=[(2,3),(1,1),(0,0)], s=0
pop (0,0): caso base, s=4 => P=[(2,3),(1,1)]
pop (1,1): caso base, s=11 => P=[(2,3)]
pop (2,3): mid=2, push (3,3) e (2,2) => P=[(3,3),(2,2)], s=11
pop (2,2): caso base, s=12 => P=[(3,3)]
pop (3,3): caso base, s=21 => P=[]
risultato 21, dimensione massima di P = 3

sumIt                = 21
sumRecHalving        = 21   profondità massima = 3
sumRecDecrease       = 21   profondità massima = 5

a = [4, 7, 1]   sumRecDecrease(a, 3)
 # | pila delle chiamate attive | azione
 1 | s(3) | n=3 != 0: chiama s(2), poi somma a[2]
 2 | s(3) s(2) | n=2 != 0: chiama s(1), poi somma a[1]
 3 | s(3) s(2) s(1) | n=1 != 0: chiama s(0), poi somma a[0]
 4 | s(3) s(2) s(1) s(0) | n=0: return 0
 5 | s(3) s(2) s(1) | return 0+a[0]=4
 6 | s(3) s(2) | return 4+a[1]=11
 7 | s(3) | return 11+a[2]=12
risultato 12, chiamate attive al massimo = 

**Prova tu.** Cambia l'array: la pila esplicita non contiene mai più di $\lceil\log_2 n\rceil+1$ coppie.

In [3]:
%%writefile prova_explicit_stack.cpp
#define NOMAIN
#include "explicit_stack.cpp"

int main()
{
    // Cambia l'array: la pila esplicita non contiene mai più di ceil(log2 n) + 1 coppie.
    vector<int> c(100);
    for (int i = 0; i < 100; i++) c[i] = i + 1;
    maxP = 0;
    long long s = SumExplicitStack(c.data(), 100);
    printf("%lld %d\n", s, maxP);
    return 0;
}

Writing prova_explicit_stack.cpp


In [4]:
!g++ -std=c++17 -O2 -o prova_explicit_stack prova_explicit_stack.cpp && ./prova_explicit_stack

5050 8


## 2. MergeSort ricorsivo e MergeSortBottomUp

**MergeSortBottomUp**$(a,n)$ — *Input*: un array $a[1..n]$ di naturali. *Output*: l'array ordinato in modo non decrescente. La stessa suddivisione di Merge Sort, ma **senza ricorsione**: prima si fondono coppie di sotto-array di lunghezza $1$, poi di lunghezza $2$, $4$, … con la procedura `Merge` del capitolo 17 (qui ripetuta). Esegue $\lceil\log_2 n\rceil$ passate e $\Theta(n\log_2 n)$ passi, come la versione ricorsiva, e non usa la pila delle chiamate.

Prima l'esempio del capitolo $a=[5,2,4,6,1,3]$, passata per passata (gli indici sono da 0). Poi il confronto, su array pseudo-casuali (stesso generatore in tutti i linguaggi, seme $12345$, valori fra $1$ e $10^6$), del numero di **confronti** fra elementi di `InsertionSort`, di `MergeSort` e di `MergeSortBottomUp`: $\Theta(n^2)$ contro $\Theta(n\log_2 n)$, con le due versioni di Merge Sort praticamente uguali ("OK" indica che i tre algoritmi danno lo stesso array ordinato), e i tempi misurati.

In [5]:
%%writefile bottom_up.cpp
#include <iostream>
#include <vector>
#include <string>
#include <algorithm>
#include <cmath>
#include <cstdio>
#include <chrono>
#include <climits>
#include <functional>
#include <array>
using namespace std;

long long comparisons = 0;   // confronti fra elementi

string Fmt(const int *a, int n)
{
    string s = "[";
    for (int i = 0; i < n; i++)
    {
        if (i > 0) s += ", ";
        s += to_string(a[i]);
    }
    return s + "]";
}

// Merge(a, l, m, r) del capitolo 17
void Merge(int *a, int l, int m, int r)
{
    int n1 = m - l + 1;
    int n2 = r - m;
    vector<int> L(n1), R(n2);
    for (int i = 0; i < n1; i++)
    {
        L[i] = a[l + i];
    }
    for (int j = 0; j < n2; j++)
    {
        R[j] = a[m + 1 + j];
    }
    int i = 0, j = 0, k = l;
    while (i < n1 && j < n2)
    {
        comparisons++;
        if (L[i] <= R[j])
        {
            a[k] = L[i];
            i++;
        }
        else
        {
            a[k] = R[j];
            j++;
        }
        k++;
    }
    while (i < n1)
    {
        a[k] = L[i];
        i++;
        k++;
    }
    while (j < n2)
    {
        a[k] = R[j];
        j++;
        k++;
    }
}

void MergeSort(int *a, int l, int r)
{
    if (l < r)
    {
        int m = l + (r - l) / 2;
        MergeSort(a, l, m);
        MergeSort(a, m + 1, r);
        Merge(a, l, m, r);
    }
}

// MergeSortBottomUp(a, n): ordina a[0..n-1] senza ricorsione
void MergeSortBottomUp(int *a, int n)
{
    int width = 1;
    while (width < n)
    {
        int l = 0;
        while (l + width < n)                   // pseudocodice: l + width <= n
        {
            int m = l + width - 1;
            int r = min(l + 2 * width - 1, n - 1);
            Merge(a, l, m, r);
            l = l + 2 * width;
        }
        width = 2 * width;
    }
}

void MergeSortBottomUpTrace(int *a, int n)
{
    int width = 1;
    int passes = 0;
    while (width < n)
    {
        int l = 0;
        string parts;
        while (l + width < n)
        {
            int m = l + width - 1;
            int r = min(l + 2 * width - 1, n - 1);
            Merge(a, l, m, r);
            if (!parts.empty()) parts += ", ";
            parts += "a[" + to_string(l) + ".." + to_string(m) + "] con a[" + to_string(m + 1) + ".." + to_string(r) + "]";
            l = l + 2 * width;
        }
        passes++;
        printf("width=%d: merge %s => %s\n", width, parts.c_str(), Fmt(a, n).c_str());
        width = 2 * width;
    }
    printf("passate = %d\n", passes);
}

// InsertionSort(a, n) (capitolo 3), con il conteggio dei confronti
void InsertionSort(int *a, int n)
{
    for (int j = 1; j < n; j++)
    {
        int key = a[j];
        int i = j - 1;
        while (i >= 0)
        {
            comparisons++;
            if (a[i] > key)
            {
                a[i + 1] = a[i];
                i = i - 1;
            }
            else
            {
                break;
            }
        }
        a[i + 1] = key;
    }
}

// Generatore congruenziale lineare: stessi numeri in Python, C++ e Java
vector<int> RandomArray(int n, unsigned long long seed = 12345)
{
    unsigned long long x = seed;
    vector<int> a(n);
    for (int i = 0; i < n; i++)
    {
        x = (x * 1103515245ULL + 12345ULL) % 2147483648ULL;
        a[i] = 1 + (int)(x % 1000000ULL);
    }
    return a;
}

double NowMs()
{
    return chrono::duration<double, milli>(chrono::steady_clock::now().time_since_epoch()).count();
}

#ifndef NOMAIN
int main()
{
    int a[] = {5, 2, 4, 6, 1, 3};
    printf("a = %s   n = %d\n", Fmt(a, 6).c_str(), 6);
    MergeSortBottomUpTrace(a, 6);

    printf("\n");
    printf("%6s %14s %10s %9s  confronti\n", "n", "InsertionSort", "MergeSort", "BottomUp");
    vector<array<double, 4>> times;
    for (int n : {1000, 2000, 4000})
    {
        vector<int> base = RandomArray(n);
        vector<int> x1 = base, x2 = base, x3 = base;
        comparisons = 0;
        double t0 = NowMs(); InsertionSort(x1.data(), n); double t1 = NowMs() - t0;
        long long c1 = comparisons;
        comparisons = 0;
        t0 = NowMs(); MergeSort(x2.data(), 0, n - 1); double t2 = NowMs() - t0;
        long long c2 = comparisons;
        comparisons = 0;
        t0 = NowMs(); MergeSortBottomUp(x3.data(), n); double t3 = NowMs() - t0;
        long long c3 = comparisons;
        vector<int> ord = base;
        sort(ord.begin(), ord.end());
        const char *ok = (x1 == x2 && x2 == x3 && x3 == ord) ? "OK" : "ERROR";
        printf("%6d %14lld %10lld %9lld  %s\n", n, c1, c2, c3, ok);
        times.push_back({(double)n, t1, t2, t3});
    }
    printf("\n");
    printf("Tempi misurati (dipendono dalla macchina):\n");
    for (auto &t : times)
    {
        printf("n=%-6d InsertionSort %10.3f ms   MergeSort %8.3f ms   MergeSortBottomUp %8.3f ms\n", (int)t[0], t[1], t[2], t[3]);
    }
    return 0;
}
#endif

Writing bottom_up.cpp


In [6]:
!g++ -std=c++17 -O2 -o bottom_up bottom_up.cpp && ./bottom_up

a = [5, 2, 4, 6, 1, 3]   n = 6
width=1: merge a[0..0] con a[1..1], a[2..2] con a[3..3], a[4..4] con a[5..5] => [2, 5, 4, 6, 1, 3]
width=2: merge a[0..1] con a[2..3] => [2, 4, 5, 6, 1, 3]
width=4: merge a[0..3] con a[4..5] => [1, 2, 3, 4, 5, 6]
passate = 3

     n  InsertionSort  MergeSort  BottomUp  confronti
  1000         248782       8716      8740  OK
  2000        1010300      19415     19438  OK
  4000        4027558      42874     42876  OK

Tempi misurati (dipendono dalla macchina):
n=1000   InsertionSort      0.078 ms   MergeSort    0.068 ms   MergeSortBottomUp    0.057 ms
n=2000   InsertionSort      0.311 ms   MergeSort    0.123 ms   MergeSortBottomUp    0.114 ms
n=4000   InsertionSort      1.189 ms   MergeSort    0.269 ms   MergeSortBottomUp    0.237 ms


**Prova tu.** Cambia $n$ (con $n=8000$ i confronti di `InsertionSort` quadruplicano rispetto a $n=4000$, quelli di Merge Sort poco più del doppio).

In [7]:
%%writefile prova_bottom_up.cpp
#define NOMAIN
#include "bottom_up.cpp"

int main()
{
    // Prova con n = 8000: i confronti di InsertionSort quadruplicano rispetto a n = 4000, quelli di Merge Sort poco più del doppio.
    int n = 8000;
    vector<int> base = RandomArray(n);
    vector<int> x1 = base, x2 = base;
    comparisons = 0;
    InsertionSort(x1.data(), n);
    long long c1 = comparisons;
    comparisons = 0;
    MergeSortBottomUp(x2.data(), n);
    printf("%lld %lld %s\n", c1, comparisons, x1 == x2 ? "True" : "False");
    return 0;
}

Writing prova_bottom_up.cpp


In [8]:
!g++ -std=c++17 -O2 -o prova_bottom_up prova_bottom_up.cpp && ./prova_bottom_up

16115652 93756 True


## 3. Potenza veloce: ricorsiva e PowerIt (square-and-multiply)

- **FastPower**$(x,n)$ (capitolo 14) — ricorsiva: $x^n=(x^{\lfloor n/2\rfloor})^2$ se $n$ è pari, $(x^{\lfloor n/2\rfloor})^2\cdot x$ se è dispari. $\Theta(\log_2 n)$ passi, $\lfloor\log_2 n\rfloor+2$ chiamate (profondità $\Theta(\log_2 n)$). Il risultato della chiamata ricorsiva è salvato in $y$ e usato due volte.
- **PowerIt**$(x,n)$ — *Input*: un numero $x$ e un naturale $n$. *Output*: $x^n$. Legge i bit di $n$ dal meno significativo: $\lfloor\log_2 n\rfloor+1$ iterazioni, memoria extra $\Theta(1)$; vale $p\cdot y^k=x^n$ a ogni iterazione.

Con $x=2$ e $n=13=(1101)_2$ si stampa la traccia del capitolo: $p$ viene moltiplicato per $x$, $x^4$ e $x^8$ (i bit di 13 uguali a 1) e $p=x^{13}$ dopo $4$ iterazioni. Poi la tabella: stesso risultato, $\lfloor\log_2 n\rfloor+1$ iterazioni contro $\lfloor\log_2 n\rfloor+2$ chiamate, contro le $n$ moltiplicazioni del ciclo naïf.

In [9]:
%%writefile power.cpp
#include <iostream>
#include <vector>
#include <string>
#include <algorithm>
#include <cmath>
#include <cstdio>
#include <chrono>
#include <climits>
#include <functional>
#include <array>
using namespace std;

int powCalls = 0;   // chiamate di FastPower
int powIters = 0;   // iterazioni di PowerIt

// FastPower(x, n): x^n per ricorsione sul dimezzamento di n
long long FastPower(long long x, int n)
{
    powCalls++;
    if (n == 0)
    {
        return 1;                   // caso base
    }
    long long y = FastPower(x, n / 2);  // una sola chiamata ricorsiva
    if (n % 2 == 0)
    {
        return y * y;               // n pari
    }
    return x * y * y;               // n dispari
}

// PowerIt(x, n): x^n con square-and-multiply
long long PowerIt(long long x, int n)
{
    long long p = 1;
    long long y = x;
    int k = n;
    while (k > 0)
    {
        powIters++;
        if (k % 2 == 1)
        {
            p = p * y;
        }
        y = y * y;
        k = k / 2;
    }
    return p;
}

void PowerItTrace(long long x, int n)
{
    string bits;
    for (int t = n; t > 0; t /= 2) bits = char('0' + t % 2) + bits;
    printf("PowerIt(%lld, %d): %d = (%s)_2\n", x, n, n, bits.c_str());
    long long p = 1, y = x;
    int k = n, it = 0;
    while (k > 0)
    {
        it++;
        const char *par = k % 2 == 1 ? "dispari" : "pari";
        if (k % 2 == 1)
        {
            p = p * y;
        }
        y = y * y;
        printf("%d. k=%d %s: p=%lld, y=%lld, k=%d\n", it, k, par, p, y, k / 2);
        k = k / 2;
    }
    printf("iterazioni = %d, risultato p = %lld\n", it, p);
}

#ifndef NOMAIN
int main()
{
    PowerItTrace(2, 13);
    printf("\n");
    printf("%3s %12s %8s %10s %6s   (iterazioni, chiamate, moltiplicazioni)\n", "n", "2^n", "PowerIt", "FastPower", "naive");
    int ns[] = {0, 1, 2, 5, 13, 30};
    for (int n : ns)
    {
        powIters = 0;
        long long r1 = PowerIt(2, n);
        powCalls = 0;
        long long r2 = FastPower(2, n);
        printf("%3d %12lld %8d %10d %6d   %s\n", n, r1, powIters, powCalls, n, (r1 == r2 && r1 == (1LL << n)) ? "OK" : "ERROR");
    }
    return 0;
}
#endif

Writing power.cpp


In [10]:
!g++ -std=c++17 -O2 -o power power.cpp && ./power

PowerIt(2, 13): 13 = (1101)_2
1. k=13 dispari: p=2, y=4, k=6
2. k=6 pari: p=2, y=16, k=3
3. k=3 dispari: p=32, y=256, k=1
4. k=1 dispari: p=8192, y=65536, k=0
iterazioni = 4, risultato p = 8192

  n          2^n  PowerIt  FastPower  naive   (iterazioni, chiamate, moltiplicazioni)
  0            1        0          1      0   OK
  1            2        1          2      1   OK
  2            4        2          3      2   OK
  5           32        3          4      5   OK
 13         8192        4          5     13   OK
 30   1073741824        5          6     30   OK


**Prova tu.** Prova un'altra base e un altro esponente (in C++ e Java il risultato deve stare in 64 bit).

In [11]:
%%writefile prova_power.cpp
#define NOMAIN
#include "power.cpp"

int main()
{
    // Prova un'altra base e un altro esponente (attenzione: in C++ e Java il risultato deve stare in 64 bit).
    powIters = 0;
    powCalls = 0;
    long long r1 = PowerIt(3, 25);
    long long r2 = FastPower(3, 25);
    printf("%lld %d %lld %d\n", r1, powIters, r2, powCalls);
    return 0;
}

Writing prova_power.cpp


In [12]:
!g++ -std=c++17 -O2 -o prova_power prova_power.cpp && ./prova_power

847288609443 5 847288609443 6


## 4. Kadane: un algoritmo iterativo migliore

**Kadane**$(a,n)$ — *Input*: un array $a[1..n]$ di $n\ge1$ interi. *Output*: la somma massima di un sottoarray non vuoto $a[l..r]$. È la versione compatta del capitolo: $cur\leftarrow\max(a[i],\,cur+a[i])$, $best\leftarrow\max(best,cur)$. Il ciclo fa $n-1$ iterazioni: $\Theta(n)$ passi e $\Theta(1)$ memoria, contro le $n(n+1)/2$ coppie $(l,r)$ della forza bruta ($\Theta(n^2)$) e contro il divide et impera ricorsivo ($\Theta(n\log_2 n)$): un buon algoritmo iterativo può battere la ricorsione.

Sull'esempio delle variazioni di prezzo del capitolo 17 il risultato è $43$. Poi, su array pseudo-casuali con valori fra $-50$ e $50$ (stesso generatore in tutti i linguaggi, seme $n$), le iterazioni di `Kadane`, le coppie esaminate dalla forza bruta e la verifica che i due algoritmi trovino la stessa somma massima; infine i tempi misurati.

In [13]:
%%writefile kadane_compact.cpp
#include <iostream>
#include <vector>
#include <string>
#include <algorithm>
#include <cmath>
#include <cstdio>
#include <chrono>
#include <climits>
#include <functional>
#include <array>
using namespace std;

string Fmt(const int *a, int n)
{
    string s = "[";
    for (int i = 0; i < n; i++)
    {
        if (i > 0) s += ", ";
        s += to_string(a[i]);
    }
    return s + "]";
}

// Kadane(a, n): somma massima di un sottoarray non vuoto di a[0..n-1]
int Kadane(int *a, int n)
{
    int best = a[0];
    int cur = a[0];                 // somma massima di un sottoarray che termina in i
    int i = 1;
    while (i < n)
    {
        cur = max(a[i], cur + a[i]);
        best = max(best, cur);
        i = i + 1;
    }
    return best;
}

// Forza bruta: somma massima e numero di coppie (l, r) esaminate
int BruteForceBest(int *a, int n, long long &pairs)
{
    int best = a[0];
    pairs = 0;
    for (int l = 0; l < n; l++)
    {
        int s = 0;
        for (int r = l; r < n; r++)
        {
            s = s + a[r];
            pairs++;
            if (s > best) best = s;
        }
    }
    return best;
}

// Interi fra -50 e 50 (generatore congruenziale lineare)
vector<int> RandomValues(int n, unsigned long long seed = 12345)
{
    unsigned long long x = seed;
    vector<int> a(n);
    for (int i = 0; i < n; i++)
    {
        x = (x * 1103515245ULL + 12345ULL) % 2147483648ULL;
        a[i] = (int)(x % 101ULL) - 50;
    }
    return a;
}

double NowMs()
{
    return chrono::duration<double, milli>(chrono::steady_clock::now().time_since_epoch()).count();
}

#ifndef NOMAIN
int main()
{
    int a[] = {13, -3, -25, 20, -3, -16, -23, 18, 20, -7, 12, -5, -22, 15, -4, 7};
    printf("a = %s\n", Fmt(a, 16).c_str());
    printf("Kadane(a, 16) = %d\n", Kadane(a, 16));
    int b[] = {-3, -1, -2, -5};
    printf("a = %s  Kadane(a, 4) = %d\n", Fmt(b, 4).c_str(), Kadane(b, 4));
    printf("\n");
    printf("%6s %7s %11s %11s %9s  stessa somma?\n", "n", "Kadane", "BruteForce", "iterations", "pairs");
    vector<array<double, 3>> times;
    for (int n : {1000, 2000, 4000})
    {
        vector<int> v = RandomValues(n, n);
        long long pairs;
        double t0 = NowMs(); int k = Kadane(v.data(), n); double t1 = NowMs() - t0;
        t0 = NowMs(); int bf = BruteForceBest(v.data(), n, pairs); double t2 = NowMs() - t0;
        printf("%6d %7d %11d %11d %9lld  %s\n", n, k, bf, n - 1, pairs, k == bf ? "OK" : "ERROR");
        times.push_back({(double)n, t1, t2});
    }
    printf("\n");
    printf("Tempi misurati (dipendono dalla macchina):\n");
    for (auto &t : times)
    {
        printf("n=%-6d Kadane %9.4f ms   BruteForce %10.3f ms\n", (int)t[0], t[1], t[2]);
    }
    return 0;
}
#endif

Writing kadane_compact.cpp


In [14]:
!g++ -std=c++17 -O2 -o kadane_compact kadane_compact.cpp && ./kadane_compact

a = [13, -3, -25, 20, -3, -16, -23, 18, 20, -7, 12, -5, -22, 15, -4, 7]
Kadane(a, 16) = 43
a = [-3, -1, -2, -5]  Kadane(a, 4) = -1

     n  Kadane  BruteForce  iterations     pairs  stessa somma?
  1000     547         547         999    500500  OK
  2000     981         981        1999   2001000  OK
  4000    1955        1955        3999   8002000  OK

Tempi misurati (dipendono dalla macchina):
n=1000   Kadane    0.0008 ms   BruteForce      0.209 ms
n=2000   Kadane    0.0014 ms   BruteForce      0.853 ms
n=4000   Kadane    0.0029 ms   BruteForce      3.385 ms


**Prova tu.** Cambia $n$: le coppie della forza bruta crescono come $n^2$, le iterazioni di Kadane come $n$.

In [15]:
%%writefile prova_kadane_compact.cpp
#define NOMAIN
#include "kadane_compact.cpp"

int main()
{
    // Prova con n = 8000: le coppie della forza bruta sono n(n+1)/2 = 32 004 000, le iterazioni di Kadane 7999.
    int n = 8000;
    vector<int> v = RandomValues(n);
    long long pairs;
    int bf = BruteForceBest(v.data(), n, pairs);
    printf("%d (%d, %lld)\n", Kadane(v.data(), n), bf, pairs);
    return 0;
}

Writing prova_kadane_compact.cpp


In [16]:
!g++ -std=c++17 -O2 -o prova_kadane_compact prova_kadane_compact.cpp && ./prova_kadane_compact

1903 (1903, 32004000)


## 5. Fibonacci: FibRec, FibMemo e FibIt (sottoproblemi che si sovrappongono)

*Input*: un naturale $n$. *Output*: il numero di Fibonacci $F(n)$, con $F(0)=0$, $F(1)=1$, $F(n)=F(n-1)+F(n-2)$.

- **FibRec**$(n)$ — traduzione diretta della definizione: i sottoproblemi si sovrappongono e le chiamate sono $2F(n+1)-1=\Theta(\varphi^n)$, esponenziali; la profondità è $n$.
- **FibMemo**$(n,memo)$ — ricorsione con memoizzazione: ogni valore è calcolato una volta (il vettore `memo` vale $-1$ per «non definito»); $2n-1$ chiamate, memoria $\Theta(n)$.
- **FibIt**$(n)$ — ciclo con due variabili: $n-1$ iterazioni, memoria $\Theta(1)$.

Sull'esempio $\texttt{FibRec}(5)$ del capitolo si conta quante volte è chiamata ogni sottoproblema (l'albero ha $15$ nodi, ma servono solo $6$ valori diversi). Poi la tabella dei conteggi (che coincidono con i valori del capitolo: $21\,891$, $242\,785$, $2\,692\,537$ chiamate per $n=20,25,30$) e i tempi misurati.

In [17]:
%%writefile fibonacci.cpp
#include <iostream>
#include <vector>
#include <string>
#include <algorithm>
#include <cmath>
#include <cstdio>
#include <chrono>
#include <climits>
#include <functional>
#include <array>
using namespace std;

long long fibCalls = 0;   // chiamate eseguite

// FibRec(n)
long long FibRec(int n)
{
    fibCalls++;
    if (n <= 1)
    {
        return n;
    }
    return FibRec(n - 1) + FibRec(n - 2);
}

// FibMemo(n, memo): memo[k] = -1 significa «non definito»
long long FibMemo(int n, long long *memo)
{
    fibCalls++;
    if (n <= 1)
    {
        return n;
    }
    if (memo[n] == -1)
    {
        memo[n] = FibMemo(n - 1, memo) + FibMemo(n - 2, memo);
    }
    return memo[n];
}

// FibIt(n)
long long FibIt(int n)
{
    if (n <= 1)
    {
        return n;
    }
    long long prev = 0;             // F(i-2)
    long long curr = 1;             // F(i-1)
    int i = 2;
    while (i <= n)
    {
        long long next = prev + curr;
        prev = curr;
        curr = next;
        i = i + 1;
    }
    return curr;
}

// Come FibRec, ma conta le chiamate per ogni valore del parametro
long long FibRecHist(int n, int *counts)
{
    counts[n]++;
    if (n <= 1)
    {
        return n;
    }
    return FibRecHist(n - 1, counts) + FibRecHist(n - 2, counts);
}

double NowMs()
{
    return chrono::duration<double, milli>(chrono::steady_clock::now().time_since_epoch()).count();
}

#ifndef NOMAIN
int main()
{
    int counts[6] = {0, 0, 0, 0, 0, 0};
    long long f5 = FibRecHist(5, counts);
    printf("FibRec(5) = %lld\n", f5);
    string line;
    int total = 0;
    for (int k = 5; k >= 0; k--)
    {
        if (k < 5) line += "  ";
        line += to_string(k) + ":" + to_string(counts[k]);
        total += counts[k];
    }
    printf("chiamate per ogni valore del parametro: %s   totale %d\n", line.c_str(), total);
    printf("\n");
    printf("%3s %8s %9s %10s %8s %6s  (chiamate, chiamate, chiamate, iterazioni)\n", "n", "F(n)", "FibRec", "2F(n+1)-1", "FibMemo", "FibIt");
    int ns[] = {10, 15, 20, 25, 30};
    for (int n : ns)
    {
        long long f = FibIt(n);
        fibCalls = 0;
        long long r1 = FibRec(n);
        long long c1 = fibCalls;
        fibCalls = 0;
        vector<long long> memo(n + 1, -1);
        long long r2 = FibMemo(n, memo.data());
        long long c2 = fibCalls;
        bool ok = (r1 == r2 && r2 == f && c1 == 2 * FibIt(n + 1) - 1 && c2 == 2 * n - 1);
        printf("%3d %8lld %9lld %10lld %8lld %6d  %s\n", n, f, c1, 2 * FibIt(n + 1) - 1, c2, n - 1, ok ? "OK" : "ERROR");
    }
    printf("\n");
    printf("Tempi misurati (dipendono dalla macchina):\n");
    for (int n : {20, 25, 30})
    {
        double t0 = NowMs(); volatile long long sink = FibRec(n); double t1 = NowMs() - t0;
        t0 = NowMs();
        for (int rep = 0; rep < 1000; rep++)
        {
            sink = FibIt(n);
        }
        double t2 = (NowMs() - t0) / 1000;
        printf("n=%-3d FibRec %10.4f ms   FibIt %10.6f ms\n", n, t1, t2);
    }
    return 0;
}
#endif

Writing fibonacci.cpp


In [18]:
!g++ -std=c++17 -O2 -o fibonacci fibonacci.cpp && ./fibonacci

FibRec(5) = 5
chiamate per ogni valore del parametro: 5:1  4:1  3:2  2:3  1:5  0:3   totale 15

  n     F(n)    FibRec  2F(n+1)-1  FibMemo  FibIt  (chiamate, chiamate, chiamate, iterazioni)
 10       55       177        177       19      9  OK
 15      610      1973       1973       29     14  OK
 20     6765     21891      21891       39     19  OK
 25    75025    242785     242785       49     24  OK
 30   832040   2692537    2692537       59     29  OK

Tempi misurati (dipendono dalla macchina):
n=20  FibRec     0.0073 ms   FibIt   0.000004 ms
n=25  FibRec     0.0948 ms   FibIt   0.000004 ms
n=30  FibRec     0.9346 ms   FibIt   0.000007 ms


**Prova tu.** Aumenta $n$ (con $n=32$ `FibRec` esegue $7\,049\,155$ chiamate, `FibMemo` $63$) e guarda come cresce il tempo di `FibRec`.

In [19]:
%%writefile prova_fibonacci.cpp
#define NOMAIN
#include "fibonacci.cpp"

int main()
{
    // Aumenta n: con n = 32 FibRec esegue 7 049 155 chiamate, FibMemo 63.
    int n = 32;
    fibCalls = 0;
    long long r1 = FibRec(n);
    printf("%lld %lld\n", r1, fibCalls);
    fibCalls = 0;
    vector<long long> memo(n + 1, -1);
    long long r2 = FibMemo(n, memo.data());
    printf("%lld %lld\n", r2, fibCalls);
    return 0;
}

Writing prova_fibonacci.cpp


In [20]:
!g++ -std=c++17 -O2 -o prova_fibonacci prova_fibonacci.cpp && ./prova_fibonacci

2178309 7049155
2178309 63


## 6. Ricorsione in coda e la sua eliminazione: GCDRec/GCDIt, RecBinarySearch/ItBinarySearch

Una chiamata ricorsiva è *in coda* se è l'ultima operazione e il suo risultato è restituito così com'è: si può sostituire con l'aggiornamento dei parametri e un salto all'inizio, cioè con un ciclo (condizione del ciclo = negazione del caso base; chiamata in coda = assegnamento simultaneo dei parametri). Il risultato è lo stesso, il numero di passi ha lo stesso ordine di grandezza e la memoria extra scende da $\Theta(\text{profondità})$ a $\Theta(1)$.

- **GCDRec**$(a,b)$ / **GCDIt**$(a,b)$ — *Input*: due naturali $a$, $b$. *Output*: il loro massimo comun divisore (algoritmo di Euclide). Con $(a,b)=(252,105)$ entrambi passano per le coppie $(252,105)$, $(105,42)$, $(42,21)$, $(21,0)$ e restituiscono $21$: quattro chiamate contro tre iterazioni. La variabile temporanea $r$ serve perché $a\bmod b$ va calcolato con il *vecchio* $a$.
- **RecBS**$(a,l,r,m)$ / **ItBS**$(a,l,r,m)$ — *Input*: un array ordinato, due indici $l$, $r$ e un valore $m$. *Output*: un indice di $m$ in $a[l..r]$ oppure $-1$. `ItBS(a, 1, n, m)` è la ricerca binaria iterativa del capitolo «Ricerca».

In [21]:
%%writefile tail_recursion.cpp
#include <iostream>
#include <vector>
#include <string>
#include <algorithm>
#include <cmath>
#include <cstdio>
#include <chrono>
#include <climits>
#include <functional>
#include <array>
using namespace std;

vector<string> gcdLog;   // coppie (a, b) visitate (solo per la stampa)
int rbsCalls = 0;        // chiamate di RecBS
int bsIters = 0;         // iterazioni di ItBS

string PairStr(long long a, long long b)
{
    return "(" + to_string(a) + "," + to_string(b) + ")";
}

// GCDRec(a, b)
long long GCDRec(long long a, long long b)
{
    gcdLog.push_back(PairStr(a, b));
    if (b == 0)
    {
        return a;
    }
    return GCDRec(b, a % b);            // chiamata in coda
}

// GCDIt(a, b)
long long GCDIt(long long a, long long b)
{
    gcdLog.push_back(PairStr(a, b));
    while (b != 0)                      // negazione del caso base
    {
        long long r = a % b;            // con il vecchio a
        a = b;
        b = r;
        gcdLog.push_back(PairStr(a, b));
    }
    return a;
}

// RecBS(a, l, r, m)
int RecBS(int *a, int l, int r, int m)
{
    rbsCalls++;
    if (l > r)
    {
        return -1;
    }
    int mid = l + (r - l) / 2;
    if (a[mid] == m)
    {
        return mid;
    }
    if (a[mid] < m)
    {
        return RecBS(a, mid + 1, r, m);     // chiamata in coda
    }
    else
    {
        return RecBS(a, l, mid - 1, m);     // chiamata in coda
    }
}

// ItBS(a, l, r, m)
int ItBS(int *a, int l, int r, int m)
{
    while (l <= r)                          // negazione del caso base l > r
    {
        bsIters++;
        int mid = l + (r - l) / 2;
        if (a[mid] == m)
        {
            return mid;                     // il secondo caso base resta nel ciclo
        }
        if (a[mid] < m)
        {
            l = mid + 1;
        }
        else
        {
            r = mid - 1;
        }
    }
    return -1;
}

string Fmt(const int *a, int n)
{
    string s = "[";
    for (int i = 0; i < n; i++)
    {
        if (i > 0) s += ", ";
        s += to_string(a[i]);
    }
    return s + "]";
}

#ifndef NOMAIN
int main()
{
    long long pairsAB[][2] = {{252, 105}, {1071, 462}, {832040, 514229}};
    for (auto &p : pairsAB)
    {
        long long a = p[0], b = p[1];
        gcdLog.clear();
        long long g1 = GCDRec(a, b);
        int calls = gcdLog.size();
        string pairs;
        for (size_t i = 0; i < gcdLog.size(); i++)
        {
            if (i > 0) pairs += " ";
            pairs += gcdLog[i];
        }
        gcdLog.clear();
        long long g2 = GCDIt(a, b);
        int iters = (int)gcdLog.size() - 1;
        printf("GCDRec(%lld, %lld) = %lld   chiamate=%d     GCDIt = %lld   iterazioni=%d   %s\n", a, b, g1, calls, g2, iters, g1 == g2 ? "OK" : "ERROR");
        if (a == 252)
        {
            printf("  coppie: %s\n", pairs.c_str());
        }
    }

    printf("\n");
    int a[] = {3, 9, 11, 12, 12, 16, 19};
    int n = 7;
    printf("a = %s   n = %d\n", Fmt(a, n).c_str(), n);
    printf("%4s %6s %6s %6s %11s\n", "m", "RecBS", "calls", "ItBS", "iterations");
    int ms[] = {12, 10, 3, 19, 100};
    for (int m : ms)
    {
        rbsCalls = 0;
        int r1 = RecBS(a, 0, n - 1, m);
        bsIters = 0;
        int r2 = ItBS(a, 0, n - 1, m);
        printf("%4d %6d %6d %6d %11d   %s\n", m, r1, rbsCalls, r2, bsIters, r1 == r2 ? "OK" : "ERROR");
    }
    return 0;
}
#endif

Writing tail_recursion.cpp


In [22]:
!g++ -std=c++17 -O2 -o tail_recursion tail_recursion.cpp && ./tail_recursion

GCDRec(252, 105) = 21   chiamate=4     GCDIt = 21   iterazioni=3   OK
  coppie: (252,105) (105,42) (42,21) (21,0)
GCDRec(1071, 462) = 21   chiamate=4     GCDIt = 21   iterazioni=3   OK
GCDRec(832040, 514229) = 1   chiamate=29     GCDIt = 1   iterazioni=28   OK

a = [3, 9, 11, 12, 12, 16, 19]   n = 7
   m  RecBS  calls   ItBS  iterations
  12      3      1      3           1   OK
  10     -1      4     -1           3   OK
   3      0      3      0           3   OK
  19      6      3      6           3   OK
 100     -1      4     -1           3   OK


**Prova tu.** Prova altre coppie per il GCD: con due numeri di Fibonacci consecutivi l'algoritmo di Euclide fa il maggior numero di passi.

In [23]:
%%writefile prova_tail_recursion.cpp
#define NOMAIN
#include "tail_recursion.cpp"

int main()
{
    // Prova altre coppie per il GCD: con due numeri di Fibonacci consecutivi l'algoritmo di Euclide fa il maggior numero di passi.
    gcdLog.clear();
    long long g = GCDRec(46368, 28657);
    printf("%lld %d\n", g, (int)gcdLog.size());
    return 0;
}

Writing prova_tail_recursion.cpp


In [24]:
!g++ -std=c++17 -O2 -o prova_tail_recursion prova_tail_recursion.cpp && ./prova_tail_recursion

1 23


## 7. Profondità della ricorsione e pila delle chiamate

La memoria usata dalla pila delle chiamate è proporzionale alla **profondità** $D(n)$ della ricorsione (il massimo numero di chiamate attive contemporaneamente), non al numero di chiamate. La tabella confronta la profondità **misurata** con la formula del capitolo:

| algoritmo | $D(n)$ |
|---|---|
| somma per decremento di uno (caso base $n=0$) | $n+1$ |
| somma per dimezzamento | $\lceil\log_2 n\rceil+1$ |
| FastPower | $\lfloor\log_2 n\rfloor+2$ |
| RecBS (caso peggiore, valore assente) | $\le\lfloor\log_2 n\rfloor+2$ |
| FibRec$(n)$, $n\ge1$ | $n$ |

Una profondità $\Theta(\log_2 n)$ non è mai un problema in pratica; una profondità $\Theta(n)$ rischia lo **stack overflow**: l'ultima parte della cella (le righe che iniziano con `~~`) dipende dal linguaggio e dalla macchina.

In [25]:
%%writefile depth.cpp
#include <iostream>
#include <vector>
#include <string>
#include <algorithm>
#include <cmath>
#include <cstdio>
#include <chrono>
#include <climits>
#include <functional>
#include <array>
using namespace std;

#include <sys/resource.h>

int depth = 0;
int maxDepth = 0;
int calls = 0;

void Enter()
{
    depth++;
    if (depth > maxDepth) maxDepth = depth;
}

long long SumRecDecrease(int *a, int n)
{
    Enter();
    long long result = (n == 0) ? 0 : SumRecDecrease(a, n - 1) + a[n - 1];
    depth--;
    return result;
}

long long SumRecHalving(int *a, int l, int r)
{
    Enter();
    long long result;
    if (l == r)
    {
        result = a[l];
    }
    else
    {
        int mid = l + (r - l) / 2;
        result = SumRecHalving(a, l, mid) + SumRecHalving(a, mid + 1, r);
    }
    depth--;
    return result;
}

long long FastPower(long long x, int n)
{
    Enter();
    long long result;
    if (n == 0)
    {
        result = 1;
    }
    else
    {
        long long y = FastPower(x, n / 2);
        result = (n % 2 == 0) ? y * y : x * y * y;
    }
    depth--;
    return result;
}

int RecBS(int *a, int l, int r, int m)
{
    calls++;
    if (l > r)
    {
        return -1;
    }
    int mid = l + (r - l) / 2;
    if (a[mid] == m)
    {
        return mid;
    }
    if (a[mid] < m)
    {
        return RecBS(a, mid + 1, r, m);
    }
    return RecBS(a, l, mid - 1, m);
}

long long FibRec(int n)
{
    Enter();
    long long result = (n <= 1) ? n : FibRec(n - 1) + FibRec(n - 2);
    depth--;
    return result;
}

int FloorLog2(int n)
{
    int f = 0;
    while ((1 << (f + 1)) <= n) f++;
    return f;
}

int CeilLog2(int n)
{
    int c = 0;
    while ((1 << c) < n) c++;
    return c;
}

#ifndef NOMAIN
int main()
{
    printf("%5s %9s %5s %8s %8s %10s %8s %6s %6s\n", "n", "decrease", "n+1", "halving", "formula", "FastPower", "formula", "RecBS", "bound");
    int ns[] = {10, 100, 500};
    for (int n : ns)
    {
        vector<int> a(n);
        for (int i = 0; i < n; i++) a[i] = i;
        maxDepth = 0; SumRecDecrease(a.data(), n); int d1 = maxDepth;
        maxDepth = 0; SumRecHalving(a.data(), 0, n - 1); int d2 = maxDepth;
        maxDepth = 0; FastPower(1, n); int d3 = maxDepth;
        calls = 0; RecBS(a.data(), 0, n - 1, n + 100); int d4 = calls;
        printf("%5d %9d %5d %8d %8d %10d %8d %6d %6d\n", n, d1, n + 1, d2, CeilLog2(n) + 1, d3, FloorLog2(n) + 2, d4, FloorLog2(n) + 2);
    }
    printf("\n");
    printf("%3s %13s\n", "n", "FibRec depth");
    int fs[] = {5, 10, 20};
    for (int n : fs)
    {
        maxDepth = 0;
        FibRec(n);
        printf("%3d %13d\n", n, maxDepth);
    }

    struct rlimit rl;
    getrlimit(RLIMIT_STACK, &rl);
    printf("~~ C++: stack limit = %lld KB; a frame of 32-48 bytes allows a depth of about %lld calls\n", (long long)rl.rlim_cur / 1024, (long long)rl.rlim_cur / 40);
    printf("~~ C++: sumRecDecrease with n = 10^7 would need about 320 MB of stack: segmentation fault (not run here)\n");
    return 0;
}
#endif

Writing depth.cpp


In [26]:
!g++ -std=c++17 -O2 -o depth depth.cpp && ./depth

    n  decrease   n+1  halving  formula  FastPower  formula  RecBS  bound
   10        11    11        5        5          5        5      5      5
  100       101   101        8        8          8        8      8      8
  500       501   501       10       10         10       10     10     10

  n  FibRec depth
  5             5
 10            10
 20            20
~~ C++: stack limit = 8192 KB; a frame of 32-48 bytes allows a depth of about 209715 calls
~~ C++: sumRecDecrease with n = 10^7 would need about 320 MB of stack: segmentation fault (not run here)


**Prova tu.** Misura la profondità per un altro $n$ (qui $2000$): `decrease` $=n+1$, `halving` $=\lceil\log_2 n\rceil+1$.

In [27]:
%%writefile prova_depth.cpp
#define NOMAIN
#include "depth.cpp"

int main()
{
    // Misura la profondità della ricorsione per un altro n (qui 2000): decrease = n+1, halving = ceil(log2 n) + 1.
    int n = 2000;
    vector<int> a(n, 1);
    maxDepth = 0;
    SumRecDecrease(a.data(), n);
    printf("%d ", maxDepth);
    maxDepth = 0;
    SumRecHalving(a.data(), 0, n - 1);
    printf("%d\n", maxDepth);
    return 0;
}

Writing prova_depth.cpp


In [28]:
!g++ -std=c++17 -O2 -o prova_depth prova_depth.cpp && ./prova_depth

2001 12
